# host

> The application under an agent, and how it says what it can do.

In [ ]:
#| default_exp host

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import ast, json, os, re, uuid
from abc import ABC, abstractmethod
from pathlib import Path
from fastcore.basics import AttrDict, ifnone, patch
from fastcore.parallel import startthread
from shalya.core import Hit, MAX_API, MAX_FILE, MAX_GREP_HITS

## Saying what a host can do

A host is asked one question over and over: can you do this? Ramabana used to ask it five ways. A
`capabilities` dict. A check for whether the subclass overrode the method. A harmless call that
catches `NotImplementedError`. A signature inspection. Thirty lines of per-group special cases
saying which of the four applied where.

One way replaces all five. Each capability group is an abstract base class carrying the name of the
group and the methods it needs. A host declares a group by inheriting it, which is a decision rather
than a coincidence: `ABCMeta` refuses to build a host that inherits a group and leaves a method
unimplemented. `provides` reads the declarations off the class.

`without` is the second half, and it is why a class alone is not enough. Whether a host can search
the web is not a property of its class. It is a property of whether `fossick` imported and whether
the session was started with the web switched off. A host that inherits `WebHost` and finds no
`fossick` adds `web` to `without`, and answers honestly for the rest of its life.

In [ ]:
#| export
class Capability(ABC):
    "One capability group. A host declares the group by inheriting the class that names it."
    group = ''

class HostError(Exception): "Something a host refuses to do, rather than a failure while doing it."

def host_err(e):
    "A caught exception, for a user-facing surface."
    return f'{type(e).__name__}: {e}'

The path boundary is not a capability. Every group needs it, so it is `Host` itself.

In [ ]:
#| export
class Host(ABC):
    "The application under an agent: the folders it may touch, and what it declares it can do."

    group = 'file'          #: every host has the path boundary the file tools need
    without = frozenset()   #: groups this instance cannot do, whatever its class declares

    @property
    def provides(self):
        "The groups this host actually supports: what its class declares, less `without`."
        declared = {c.group for c in type(self).__mro__ if getattr(c, 'group', '')}
        return declared - set(self.without)

    def can(self, group):
        "Whether this host supports one group."
        return group in self.provides

    @property
    @abstractmethod
    def roots(self):
        "The open folders, as absolute paths. The agent is told about these and confined to them."

    @property
    def added_roots(self):
        "The roots opened after this host was built, which a resumed session must not inherit."
        return []

    def add_root(self, path):
        "Open another folder, and return it resolved. Widening the write boundary, so hosts may refuse."
        raise HostError('this host cannot open another folder')

    @abstractmethod
    def check(self, path, must_exist=False, reading=False):
        """The single chokepoint: resolve `path`, refuse anything outside `roots`, return a `Path`.

        `reading=True` says the caller will only *read* what comes back, and it is the one case a
        host may answer for a path outside `roots`. See `LocalHost(read_outside=)`.
        """

    @abstractmethod
    def walk(self):
        "Every readable file under the open folders."

    @abstractmethod
    def read(self, path):
        "One file's text, or None when it cannot be read."

    @abstractmethod
    def write(self, path, text):
        "Write `text` to `path`, through the same sandbox `check` enforces. Returns the path written."

    @abstractmethod
    def text_at(self, path):
        "One file as a single diffable document, `''` when it does not exist yet, None on error."

    @property
    def approvals(self):
        "Where a write goes to be approved. Shalya never reads this. Ramabana and Leela both set it."
        return None

    def note(self, text):
        "Tell the user something out of band. Never blocks. A host may drop it."
        pass

## The groups

Nine of them. Each names its group and the methods that group's tools call, and nothing else: no
bodies, no state, no ordering between them. This is the whole contract between a host and the
toolset.

In [ ]:
#| export
class CodeHost(Capability):
    "Reading the shape of a codebase rather than its bytes."
    group = 'code'

    @abstractmethod
    def search(self, query, limit=20):
        "Search the code index for `query`, returning `Hit`s. Semantic if an index exists, literal if not."

    @abstractmethod
    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."

    @abstractmethod
    def peers(self, path, line, limit=20):
        "Code shaped like whatever is defined at `path`:`line`. Every place a pattern was already used."

    @abstractmethod
    def public_api(self, package, limit=MAX_API):
        """Every public name `package` exports, as `Hit`s whose `symbol` is the qualified name.

        Raise rather than return `[]` when there is no index. "This package exports nothing" and
        "nothing could be looked up" must not reach the model as the same answer.
        """

    def grep(self, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
        "Every line matching `pattern` exactly, as `Hit`s. None means this host has no exact matcher."
        return None

    @property
    def indexed(self):
        "The folders whose index is built and searchable now. Empty means `public_api` has nothing to read."
        return ()

    @property
    def search_note(self):
        "Which engine answered, and why. Shown when a search finds nothing."
        return ''


class WebHost(Capability):
    "Going out to the network now, as opposed to recalling what was read before."
    group = 'web'

    @abstractmethod
    def web_search(self, query, n=20):
        "Search the web. Returns objects with `.title` and `.url`."

    @abstractmethod
    def read_url(self, url, remember=True):
        "One page as markdown. `remember=False` keeps sensitive or low-quality results ephemeral."

    @abstractmethod
    def research(self, query):
        "Search and read the top results into one cited digest. Slower than `web_search`."

    @property
    def research_note(self): return ''


class NotebookHost(Capability):
    """Notebooks, as far as a host has to know about them.

    Shalya owns no notebook representation. Exhash addresses cells by path and id without one, so
    only the two operations that need to know what a notebook *is* are delegated.
    """
    group = 'notebook'

    @abstractmethod
    def nb_cells(self, path):
        "`[(id, cell_type, source)]` for one notebook."

    @abstractmethod
    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        "Insert a cell (-1 appends), creating the notebook if needed. Returns the new cell's id."


class MemoryHost(Capability):
    "What was read before, kept as a tree rather than as a transcript."
    group = 'memory'

    @abstractmethod
    def memory_search(self, query, limit=8):
        "Search remembered pages as whole tree sections, returning structured rows."

    @abstractmethod
    def memory_tree(self, document=''):
        "The heading tree for remembered documents. An empty document lists every root."

    @abstractmethod
    def memory_read(self, node_id):
        "Read one remembered section and its children by stable node id."

    @abstractmethod
    def memory_topics(self, limit=12):
        "Labelled semantic clusters across remembered research."

    @abstractmethod
    def memory_forget(self, doc_id):
        "Purge one remembered document and all derived tree, chunk and vector data."

    @abstractmethod
    def remember(self, text, title=None, tags=()):
        "File `text` into durable memory as a note. Returns the document record."

    @abstractmethod
    def ask(self, question, ref=None, instruction='', **kw):
        "Answer `question` out of remembered research, with citations, as a dict."


class WatchHost(Capability):
    """What the agent arranged to read later.

    A watch is a job the host re-runs on an interval, `poll` is the tick, and a reminder is a watch
    that files its own text.
    """
    group = 'watch'

    @abstractmethod
    def watch(self, target, action='remind', every='1d', note=None, **params):
        "Register a recurring job. `target` is a URL, a query, or the text of a reminder."

    @abstractmethod
    def watches(self, due_only=False):
        "Every registered watch, soonest first. `due_only` keeps the ones that have come due."

    @abstractmethod
    def unwatch(self, watch_id):
        "Delete one watch. Whatever it already filed stays in memory."

    @abstractmethod
    def poll(self):
        "Run every watch that is due and report what fired. One failing watch must not stop the rest."

    @property
    def watch_actions(self):
        "The `action` values this host's `watch` will accept."
        return ('remind',)


class SessionHost(Capability):
    "The live namespace, and the terminal beside it."
    group = 'session'

    @abstractmethod
    def run_python(self, code):
        """Run `code` in the user's live namespace under whatever restrictions the host imposes.

        The contract the agent is briefed on, and the host's to enforce: read anything, bind
        results to new names, never rebind or delete the owner's.
        """

    @abstractmethod
    def inspect_python(self, code, scope='isolated'):
        """Run `code` against the live namespace without touching what the user has.

        Two scopes, both protecting the owner's variables, by different means:

        - `'isolated'` runs in an allowlist sandbox on a *copy*. Attribute reads and builtins
          work. Most library method calls are refused. The default, and it needs no trust.
        - `'overlay'` runs the real interpreter against the real namespace under an AST policy:
          read anything, bind names in the agent's own layer, never delete, rebind or mutate
          the owner's. `list(df.columns)` works here. In the sandbox it does not.

        A host may refuse `'overlay'`. See `scopes`.
        """

    @abstractmethod
    def list_vars(self):
        "What is in the live namespace: name, type, and a short value, one per line."

    def terminal_text(self, lines=200):
        "What the IDE's terminal has printed. Read-only: it shows what the user ran, it cannot run anything."
        return ''

    @property
    def scopes(self):
        "The scopes `inspect_python` will actually honour, most trusted last."
        return ('isolated',)

    @property
    def kernel_kind(self):
        "What runs the live namespace. `'ipymini'` inspects while a cell is busy. Anything else queues."
        return 'ipykernel'

    @property
    def concurrent(self): return self.kernel_kind == 'ipymini'


class ShellHost(Capability):
    "Running a command on the machine."
    group = 'shell'

    @abstractmethod
    def run_cmd(self, command, cwd=None, timeout=120):
        """Run `command` in a shell and return `(exit_code, combined_output)`.

        The contract a host must keep, because the tool trusts it:

        - `cwd` is resolved through `check`. Confining the *working directory* is not confining
          the command, which is why `run_shell` is a write tool and goes to a person.
        - stdout and stderr come back interleaved, in one string, in order.
        - `timeout` is enforced and the whole process *group* is killed on expiry.
        - A failed command returns a non-zero exit code rather than raising.
        """

    @property
    def shell_note(self):
        "How commands are run here, or why they are not."
        return ''


class ApiHost(Capability):
    "Reading an API specification and calling what it describes."
    group = 'api'

    @abstractmethod
    def api_load(self, src, name=''):
        "Load an OpenAPI specification from a path or a URL. Returns what it is now called."

    @abstractmethod
    def api_ops(self, group='', name='', match='', limit=None, offset=0):
        "The operations a loaded specification describes, filtered and paged."

    @abstractmethod
    def api_count(self, group='', name='', match=''):
        "How many operations that filter matches, without listing them."

    @abstractmethod
    def api_call(self, operation, name='', **params):
        "Call one operation from a loaded specification."


class GitHost(Capability):
    """A working tree the git tools can act on.

    The group needs no methods. `gheasy` reaches the repository through `roots`, so declaring the
    group is the whole contract.
    """
    group = 'git'

`provides` is the answer to every "can you?" the toolset asks. Read it against a host that
declares two groups and has lost one of them.

In [ ]:
class Demo(Host, CodeHost, WebHost):
    without = {'web'}                                    # fossick is not installed here
    @property
    def roots(self): return ['/proj']
    def check(self, path, must_exist=False, reading=False): return Path('/proj')/path
    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): return str(path)
    def text_at(self, path): return ''
    def search(self, query, limit=20): return [Hit('/proj/a.py', 1, 'a', 'def a(): ...')]
    def symbols(self, path): return []
    def peers(self, path, line, limit=20): return []
    def public_api(self, package, limit=MAX_API): return []
    def web_search(self, query, n=20): return []
    def read_url(self, url, remember=True): return None
    def research(self, query): return ''

d = Demo()
sorted(d.provides), d.can('code'), d.can('web')

In [ ]:
test_eq(sorted(d.provides), ['code', 'file'])   # every host has the file group
test_eq(d.can('code'), True)
test_eq(d.can('web'), False)
test_eq(d.can('memory'), False)

Declaring a group and leaving a method out is refused where the host is built, naming the method.
This is what makes the declaration worth trusting: a host cannot claim a group it has not written.

In [ ]:
class Broken(Demo, NotebookHost):
    def nb_cells(self, path): return []                  # and no `nb_add_cell`

test_fail(Broken, contains='nb_add_cell')

## A host over real folders

`LocalHost` is the reference implementation: enough of a host to run an agent from a terminal, an
MCP server or a test. It declares every group, and drops the ones its optional dependencies or its
construction arguments say it cannot do. One class covers every combination of groups.
`mk_host(vault=True, spec=True)` sets two arguments where it used to synthesise a `VaultSpecHost` at
runtime. Adding a kernel to a running session becomes an assignment rather than a replacement.

In [ ]:
#| export
SANDBOX = 'path is outside the open folders'
SECRET = 'path holds credentials and is never read'
NO_ROOTS = 'no folders are open, so no path is inside them'

#: Never opened, even with `read_outside`. A read tool is not a way to exfiltrate a key.
DENY = ('*/.ssh/*', '*/.aws/*', '*/.gnupg/*', '*/.config/gcloud/*', '*/.netrc',
        '*/.git-credentials', '*/.docker/config.json', '*/.kube/config', '*/.npmrc', '*/.pypirc')

SKIP_DIRS = frozenset({'.git', '.hg', '.svn', '__pycache__', '.venv', 'venv', 'node_modules',
                       '.mypy_cache', '.pytest_cache', '.ipynb_checkpoints', 'dist', 'build'})
SKIP_SUFFIXES = frozenset({'.pyc', '.pyo', '.so', '.dylib', '.dll', '.a', '.o', '.zip', '.gz',
                           '.tar', '.jpg', '.jpeg', '.png', '.gif', '.pdf', '.mp4', '.mp3'})
MAX_VARS = 200
LD_CHARS = 4000           # of a page's JSON-LD to keep. Enough for a product, not a catalogue

_LD = re.compile(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', re.S | re.I)

In [ ]:
#| export
def denied(path, patterns=DENY):
    "Whether `path` is one of the things reading outside the open folders still must not open."
    from fnmatch import fnmatch
    s = Path(path).as_posix()
    return any(fnmatch(s, pat) for pat in patterns)

def _md_doc(d):
    "One of fossick's document readers' results as markdown: the fields it has, then its text."
    if isinstance(d, str): return d
    if not isinstance(d, dict): return str(d or '')
    head = [f'**{k}**: {v}' for k in ('title', 'authors', 'published', 'channel', 'duration', 'link')
            if (v := d.get(k)) not in (None, '', [], {})]
    body = next((str(d[k]) for k in ('source', 'text', 'content', 'summary') if d.get(k)), '')
    return '\n'.join(head + [''] + [body]).strip() if head else body.strip()

def _fuse(legs, limit):
    "Merge ranked `Hit` lists with `litesearch.rrf_all`. Identity is `path:line`."
    legs = [list(l) for l in legs if l]
    if not legs: return []
    if len(legs) == 1: return legs[0][:limit]
    from litesearch import rrf_all   # imported here: it pulls pandas, and only fusion needs it
    by_key, lists = {}, []
    for leg in legs:
        rows = []
        for h in leg:
            key = f'{h.path}:{h.line}'
            by_key.setdefault(key, h)
            rows.append({'_fid': key})
        lists.append(rows)
    try: fused = rrf_all(lists, id_key='_fid', limit=limit)
    except Exception: return legs[0][:limit]   # a bad fusion degrades the ranking, never `search`
    return [by_key[r['_fid']] for r in fused if r.get('_fid') in by_key]

def ld_json(html):
    "The `schema.org` JSON-LD blocks in `html`. Where a page states its price, author or rating."
    out = []
    for m in _LD.finditer(html or ''):
        try: out.append(json.loads(m.group(1)))
        except Exception: pass
    return out

Construction opens the folders and starts one Kosha sync per root in a daemon thread. It starts
before the first turn, so the index is usually ready by the time a search asks for it, and `search`
answers from ripgrep while it is not.

`without` is filled here, once, from what actually imported and what the caller asked for. Nothing
recomputes it later.

In [ ]:
#| export
class LocalHost(Host, CodeHost, WebHost, NotebookHost, SessionHost, ShellHost, MemoryHost,
                WatchHost, ApiHost, GitHost):
    "The reference `Host`: enough of one to run an agent from a terminal, an MCP server or a test."

    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 ns=None,               # the live namespace. A fresh dict when None
                 approvals=None,        # an `Approvals`, or None to gate nothing
                 note=None,             # callable for out-of-band status lines
                 web=True,              # wire the web tools to fossick when it is installed
                 index=True,            # start a Kosha sync for every open root
                 graph=False,           # build Kosha's call graph during sync
                 rerank=True,           # reorder Kosha's hits with its flashrank cross-encoder
                 rerank_model=None,     # flashrank model name. None is its fast default
                 vault=None,            # a vishalakshi store, for the memory and watch groups
                 apis=None,             # whatever answers the api group, for the api group
                 read_outside=False,    # let read-only tools name any path on this machine
                 deny=DENY):            # what `read_outside` still refuses to open
        self._roots = [str(Path(r).expanduser().resolve()) for r in roots]
        self._added_roots = []         # opened later, and not inherited by a resumed session
        self.ns = ifnone(ns, {'__name__': '__main__'})
        self._approvals, self._note, self.web = approvals, note, web
        self.read_outside, self.deny = bool(read_outside), tuple(deny or ())
        self.transcript = []           # what this process has printed, for `read_terminal`
        self._indexes, self._index_errors, self._index_thread = [], [], None
        self._pending = list(self._roots)     # roots whose sync has not returned yet
        self.rerank, self.rerank_model, self._rerank_note = bool(rerank), rerank_model, ''
        self.graph, self.vault, self.apis = graph, vault, apis
        self.without = self._absent()
        if index: self.sync_index()

    def _absent(self):
        "The groups this host cannot do: no backend attached, or the package never imported."
        out = set()
        if not self.web or not _installed('fossick'): out.add('web')
        if self.vault is None: out |= {'memory', 'watch'}
        if self.apis is None: out.add('api')
        return frozenset(out)

In [ ]:
#| export
def _installed(mod):
    "Whether `mod` imports. Asked once per host, at construction, and never again."
    from importlib.util import find_spec
    try: return find_spec(mod) is not None
    except Exception: return False

### The path boundary

`check` is the reason this class exists. It resolves a path before comparing it, so `..` and a
symlink pointing out of the tree are both refused rather than followed.

In [ ]:
#| export
@patch(as_prop=True)
def roots(self:LocalHost): return list(self._roots)

@patch(as_prop=True)
def added_roots(self:LocalHost):
    "Roots opened after construction. `/resume` lapses these, and says that it did."
    return list(self._added_roots)

@patch
def add_root(self:LocalHost, path):
    """Open another folder for reading and writing, and index it. Returns it resolved.

    The one operation that widens the write boundary of a running session, so it refuses
    anything that is not already a directory rather than creating one.
    """
    p = Path(path).expanduser().resolve()
    if not p.exists(): raise HostError(f'no such folder: {p}')
    if not p.is_dir(): raise HostError(f'not a folder, so it cannot be a root: {p}')
    if str(p) in self._roots: return str(p)
    self._roots.append(str(p)); self._added_roots.append(str(p))
    self._pending.append(str(p))
    try: self.sync_index()
    except Exception as e: self._index_errors.append(host_err(e))
    return str(p)

@patch
def check(self:LocalHost, path, must_exist=False, reading=False):
    "Resolve `path`. Refuse outside `roots` (unless `read_outside` and `reading`). Walks stay confined."
    p = Path(path).expanduser()
    if not self._roots: raise HostError(f'{NO_ROOTS}: {p}')  # empty roots must refuse, not IndexError
    if not p.is_absolute(): p = Path(self._roots[0])/p
    p = p.resolve()  # collapse `..` and out-of-root symlinks before comparing
    if not any(p == Path(r) or Path(r) in p.parents for r in self._roots):
        if not (reading and self.read_outside): raise HostError(f'{SANDBOX}: {p}')
        if denied(p, self.deny): raise HostError(f'{SECRET}: {p}')
    if must_exist and not p.exists(): raise HostError(f'no such file: {p}')
    return p

@patch(as_prop=True)
def roots_note(self:LocalHost):
    "How paths are resolved here, in one line, for the briefing and a status bar."
    n = len(self._roots)
    return (f'{n} folder(s); reads may name any path on this machine, writes may not'
            if self.read_outside else f'{n} folder(s); nothing outside them is readable')

@patch
def _walk(self:LocalHost, root):
    "Files under `root`, skipping the same generated dirs/suffixes `grep` covers."
    try:
        from rgapi import fd
        rows = fd(root=root, skip_dir=sorted(SKIP_DIRS), max_filesize=MAX_FILE,
                  exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
        for p in rows:
            p = Path(p)
            if not p.is_absolute(): p = Path(root)/p
            if p.is_symlink() or not p.is_file(): continue
            yield p
        return
    except Exception: pass
    for p in sorted(Path(root).rglob('*')):
        if any(part in SKIP_DIRS for part in p.parts): continue
        if not p.is_file() or p.is_symlink(): continue
        if p.suffix.lower() in SKIP_SUFFIXES: continue
        try:
            if p.stat().st_size > MAX_FILE: continue
        except OSError: continue
        yield p

@patch
def walk(self:LocalHost):
    return [p for r in self._roots for p in self._walk(r)]

@patch
def read(self:LocalHost, path):
    try: return self.check(path, must_exist=True, reading=True).read_text(encoding='utf-8')
    except Exception: return None

@patch
def write(self:LocalHost, path, text):
    p = self.check(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(str(text), encoding='utf-8')
    return str(p)

@patch
def text_at(self:LocalHost, path):
    "One file as a diffable document: a notebook as its cell sources, anything else as text."
    try: p = self.check(path)
    except Exception: return None
    if not p.exists(): return ''
    if p.suffix == '.ipynb':
        try:
            from fastcore.nbio import read_nb
            return '\n\n'.join(''.join(c.source) for c in read_nb(p).cells)
        except Exception: return None
    try: return p.read_text(encoding='utf-8')
    except Exception: return None

@patch(as_prop=True)
def approvals(self:LocalHost): return self._approvals

@patch
def note(self:LocalHost, text):
    self.transcript.append(str(text))
    if self._note:
        try: self._note(str(text))
        except Exception: pass

### Seeing the code

Three engines answer a search, and which one answered is part of the result. Kosha's hybrid index
when it has synced, ripgrep when it has not, and reading the files when neither is installed.

In [ ]:
#| export
@patch
def sync_index(self:LocalHost, wait=False, force=False):
    "Run `Kosha.sync` for every open root, once, in a daemon thread. Each root publishes as it returns."
    if self._index_thread is None or not self._index_thread.is_alive():
        def run():
            try:
                os.environ.setdefault('TQDM_DISABLE', '1')  # kosha's tqdm even with verbose=False
                from kosha import Kosha
            except Exception as e:
                self._index_errors.append(host_err(e)); self._pending = []; return
            for root in list(self._roots):
                try:
                    k = Kosha(dir=Path(root), busy_timeout=30000)
                    k.sync(dir=Path(root), verbose=False, force=force, pyproject=True, graph=self.graph)
                    self._indexes.append(k)
                except Exception as e: self._index_errors.append(host_err(e))
                finally:
                    try: self._pending.remove(root)
                    except ValueError: pass
        self._index_thread = startthread(run, daemon=True)
        self._index_thread.name = 'shalya-kosha-sync'
    if wait: self._index_thread.join()
    return self

@patch(as_prop=True)
def index_ready(self:LocalHost):
    "Whether *every* open folder is indexed. `indexed` is the per-folder answer `search` uses."
    return bool(self._indexes) and not self._pending

@patch(as_prop=True)
def indexed(self:LocalHost):
    "The folders whose index is built and searchable now. The rest are still syncing."
    return [str(getattr(k, 'root', '')) for k in list(self._indexes)]

@patch
def wait_index(self:LocalHost, timeout=None):
    "Wait for the automatic Kosha sync. Returns whether semantic search is ready."
    if self._index_thread is not None: self._index_thread.join(timeout)
    return self.index_ready

@patch
def _rg(self:LocalHost, query, limit, regex=False, ignore_case=False, path_filter='', per_file=5, every_file=False):
    "Search through `rgapi.rg`. `every_file=True` matches what `walk` yields, hidden files included."
    try: from rgapi import rg
    except Exception: return None
    pattern = query if regex else re.escape(query)
    kw = dict(case_sensitive=(False if ignore_case else None), smart_case=not ignore_case,
              max_filesize=MAX_FILE, timeout_ms=20_000)
    if path_filter: kw['glob'] = f'*{path_filter}*'
    if every_file:
        kw.update(hidden=True, ignore=False, skip_dir=sorted(SKIP_DIRS),
                  exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
    hits, counts = [], {}
    try:
        for root in self._roots:
            # pull enough rows to honour per-file caps, then trim to `limit`
            pull = None if not per_file else max(limit * 8, limit)
            for m in rg(pattern, root=root, max_results=pull, **kw):
                if getattr(m, 'kind', 'match') != 'match': continue
                path = Path(m.path)
                if not path.is_absolute(): path = Path(root)/path
                path_s = str(path)
                if per_file:
                    n = counts.get(path_s, 0)
                    if n >= per_file: continue
                    counts[path_s] = n + 1
                hits.append(Hit(path_s, int(m.line_number), '', (m.line or '').strip()[:200]))
                if len(hits) >= limit: return hits
    except Exception: return None
    return hits

@patch
def grep(self:LocalHost, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
    "Exact matching through ripgrep. None when `rgapi` is unavailable. The tool then reads files itself."
    return self._rg(pattern, limit, regex=regex, ignore_case=ignore_case,
                    path_filter=path_filter, per_file=None, every_file=True)

@patch
def _ranked(self:LocalHost, call, **kw):
    "One Kosha context call, reordered by its cross-encoder when reranking is on and working."
    if self.rerank:
        try: return call(rerank=True, rerank_model=self.rerank_model, **kw)
        except Exception as e:   # flashrank fetches its model on first use. Fall back once
            self.rerank = False
            self._rerank_note = f'; reranking off ({host_err(e)})'
    return call(**kw)

@patch
def _semantic(self:LocalHost, query, limit):
    "Kosha hybrid results (repo + env + graph) as the Host's stable `Hit` shape."
    indexes = list(self._indexes)
    if not indexes: return []
    out, seen = [], set()
    for k in indexes:
        try:
            rows = self._ranked(k.context, q=query, limit=limit, repo=True, env=True,
                                graph=self.graph, columns='content,metadata')
        except Exception as e:
            self._index_errors.append(host_err(e)); continue
        for row in rows:
            row = dict(row)
            meta = row.get('metadata') or {}
            if isinstance(meta, str):
                try: meta = ast.literal_eval(meta)
                except Exception: meta = {}
            path = str(meta.get('path') or row.get('path') or '')
            line = int(meta.get('lineno') or 1)
            key = (path, line)
            if key in seen: continue
            seen.add(key)
            symbol = meta.get('mod_name') or meta.get('name') or ''
            text = ' '.join(str(row.get('content') or '').split())[:240]
            out.append(Hit(path, line, str(symbol), text))
            if len(out) >= limit: return out
    return out

@patch
def _scan(self:LocalHost, query, limit):
    "Every matching line, by reading the files. What is left when there is no index and no ripgrep."
    hits = []
    for p in self.walk():
        try: text = p.read_text(encoding='utf-8')
        except Exception: continue
        if query not in text: continue
        for i, line in enumerate(text.splitlines(), 1):
            if query in line:
                hits.append(Hit(str(p), i, '', line.strip()[:200]))
                if len(hits) >= limit: return hits
    return hits

@patch
def search(self:LocalHost, query, limit=20):
    "The code index and the literal scan, fused by rank rather than tried in order."
    if not (query or '').strip(): return []
    rg = self._rg(query, limit)
    if (hits := _fuse([self._semantic(query, limit), rg or []], limit)): return hits
    return [] if rg is not None else self._scan(query, limit)

@patch(as_prop=True)
def search_note(self:LocalHost):
    n, tot = len(self._indexes), len(self._roots)
    if n:
        where = f'{n} of {tot} folder(s)' if self._pending else f'{tot} folder(s)'
        return f'Kosha semantic + keyword index over {where} and environment fused with ripgrep{self._rerank_note}'
    if self._index_errors: return f'Kosha unavailable ({self._index_errors[-1]}); literal fallback'
    return 'Kosha sync in progress; literal fallback via ripgrep'

@patch
def public_api(self:LocalHost, package, limit=MAX_API):
    "Kosha's public surface for `package`, `@patch`-added methods included."
    if not str(package or '').strip(): return []      # the capability probe
    indexes = list(self._indexes)
    if not indexes: raise HostError(f'no code index: {self.search_note}')
    out, seen = [], set()
    for k in indexes:
        try: rows = k.public_api(package, meta_cols='name,mod_name,docstring,path,lineno', limit=limit)
        except Exception as e: self._index_errors.append(host_err(e)); continue
        for row in rows:
            row = dict(row)
            name = str(row.get('mod_name') or row.get('name') or '')
            if not name or name in seen: continue
            seen.add(name)
            doc = ' '.join(str(row.get('docstring') or '').split())[:200]
            out.append(Hit(str(row.get('path') or ''), int(row.get('lineno') or 1), name, doc))
            if len(out) >= limit: return out
    return out

@patch
def _defs(self:LocalHost, path):
    "Every def/class in one file as `(line, qualified_name, depth)`, by parsing rather than grepping."
    src = self.read(path)
    if src is None: return []
    try: tree = ast.parse(src)
    except SyntaxError: return []
    out = []
    def walk(node, prefix='', depth=0):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                name = f'{prefix}{child.name}'
                out.append((child.lineno, name, depth))
                walk(child, f'{name}.', depth + 1)
    walk(tree)
    return out

@patch
def symbols(self:LocalHost, path):
    "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
    p = self.check(path, reading=True)
    out = []
    for line, name, depth in self._defs(p):
        h = Hit(str(p), line, name, '')
        h.score = depth
        out.append(h)
    return out

@patch
def peers(self:LocalHost, path, line, limit=20):
    "Every other place the symbol defined at `path`:`line` is mentioned."
    p = self.check(path, reading=True)
    defs = self._defs(p)
    name = next((n for ln, n, _ in sorted(defs, key=lambda d: -d[0]) if ln <= int(line)), None)
    if name is None: return []
    leaf = name.split('.')[-1]
    return [h for h in self.search(leaf, limit * 2)
            if not (str(h.path) == str(p) and h.line == int(line))][:limit]

### Notebooks, the session and the shell

In [ ]:
#| export
@patch
def nb_cells(self:LocalHost, path):
    from fastcore.nbio import read_nb
    nb = read_nb(self.check(path, must_exist=True, reading=True))
    return [(c.get('id', ''), c.cell_type, ''.join(c.source)) for c in nb.cells]

@patch
def nb_add_cell(self:LocalHost, path, source, index=-1, cell_type='code'):
    from fastcore.nbio import read_nb, write_nb, mk_cell, dict2nb
    p = self.check(path)
    nb = read_nb(p) if p.exists() else dict2nb({'cells': [], 'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5})
    cell = mk_cell(source, cell_type)
    if not cell.get('id'): cell['id'] = uuid.uuid4().hex[:8]
    nb.cells.append(cell) if index < 0 else nb.cells.insert(int(index), cell)
    p.parent.mkdir(parents=True, exist_ok=True)
    write_nb(nb, p)
    return cell['id']

@patch
def _exec(self:LocalHost, code, ns):
    "Run `code` in `ns`, returning printed output plus the last expression's value."
    import contextlib, io
    buf = io.StringIO()
    tree = ast.parse(str(code))
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        if tree.body: exec(compile(tree, '<agent>', 'exec'), ns)
        value = eval(compile(ast.Expression(last.value), '<agent>', 'eval'), ns) if last else None
    out = buf.getvalue()
    if value is not None: out += ('' if not out or out.endswith('\n') else '\n') + repr(value)
    return out.strip() or '(no output)'

@patch
def run_python(self:LocalHost, code):
    "Run `code` in the live namespace. Failures come back as text: a tool cannot usefully raise."
    try: return self._exec(code, self.ns)
    except Exception as e: return f'{host_err(e)}'

@patch
def inspect_python(self:LocalHost, code, scope='isolated'):
    "Run `code` against a *copy* of the namespace. Nothing the user made can move."
    if scope not in self.scopes: return f'this host only honours {self.scopes}'
    try: return self._exec(code, dict(self.ns))
    except Exception as e: return f'{host_err(e)}'

@patch(as_prop=True)
def scopes(self:LocalHost):
    "Isolated only. Overlay needs an AST policy over the real namespace, which belongs to an IDE."
    return ('isolated',)

@patch(as_prop=True)
def kernel_kind(self:LocalHost): return 'inprocess'

@patch
def list_vars(self:LocalHost):
    rows = []
    for k, v in list(self.ns.items())[:MAX_VARS]:
        if k.startswith('_') or callable(v) or isinstance(v, type(ast)): continue
        try: short = repr(v)
        except Exception: short = '<unreprable>'
        rows.append(f'{k:20} {type(v).__name__:12} {short[:60]}')
    return '\n'.join(rows)

@patch
def terminal_text(self:LocalHost, lines=200):
    "What this process has printed, when the application records it in `transcript`."
    return '\n'.join(str(x) for x in self.transcript[-int(lines):])

@patch
def run_cmd(self:LocalHost, command, cwd=None, timeout=120):
    """Run `command` in a shell under one of the open folders.

    Started in its own process group, and the *group* is killed on timeout. A command
    that spawns children cannot leave one behind. Stdout and stderr are interleaved.
    """
    import subprocess
    if not str(command or '').strip(): return 0, ''   # the capability probe
    if not (cwd or self._roots): raise HostError(NO_ROOTS)
    d = self.check(cwd) if cwd else Path(self._roots[0])
    if not d.is_dir(): raise HostError(f'not a directory: {d}')
    p = subprocess.Popen(str(command), shell=True, cwd=str(d), text=True, errors='replace',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         start_new_session=True)
    try: out, _ = p.communicate(timeout=max(1, int(timeout)))
    except subprocess.TimeoutExpired:
        import os, signal
        try: os.killpg(p.pid, signal.SIGKILL)
        except Exception: p.kill()
        out, _ = p.communicate()
        return 124, (out or '') + f'\n[killed after {int(timeout)}s]'
    return p.returncode, out or ''

@patch(as_prop=True)
def shell_note(self:LocalHost):
    return f'shell, in {self._roots[0]}' if self._roots else 'no folder to run a command in'

### The web

Every page arrives as markdown. A URL that is a paper, a repository file or a video gets the reader
that knows its shape. Everything else is fetched. A page that comes back too thin to be real is
fetched again the expensive way.

In [ ]:
#| export
@patch
def _fossick(self:LocalHost):
    if not self.web: raise NotImplementedError
    try:
        import fossick
        return fossick
    except Exception: raise NotImplementedError

@patch
def web_search(self:LocalHost, query, n=20):
    "Search the web through fossick. An empty query answers `[]`: that is how `tools_for` probes."
    fossick = self._fossick()
    if not str(query).strip(): return []
    rows = fossick.search(str(query), n=int(n))   # `n` to fossick. Its own default is 10
    return [AttrDict(title=str(r.get('title', '')), url=str(r.get('href') or r.get('url', ''))) for r in rows]

@patch
def read_url(self:LocalHost, url, remember=True):
    "Page as markdown via fossick: `READERS`, then `fetch(auto=True)`, thin-page escalate, JSON-LD."
    fossick = self._fossick()
    for rx, name, kw in self.READERS:
        if not rx.search(str(url)) or (reader := getattr(fossick, name, None)) is None: continue
        try: text = _md_doc(reader(str(url), **kw))
        except Exception as e:   # a reader that cannot answer is not a URL that cannot be read
            self.note(f'{name} could not read {url} ({host_err(e)}); fetching the page')
            break
        if text.strip(): return AttrDict(text=text, url=str(url))
        break
    page = fossick.fetch(str(url), auto=True)
    text = str(fossick.to_md(page) or '') if page is not None else ''
    if len(text.strip()) < self.THIN_PAGE:
        for opts in ({'heavy': True}, {'stealthy': True}):
            try: heavy = fossick.fetch(str(url), **opts)
            except Exception: continue
            if len((got := str(fossick.to_md(heavy) or '')).strip()) >= self.THIN_PAGE:
                page, text = heavy, got
                break
    if (ld := ld_json(getattr(page, 'html_content', '') or '')):
        text = f'<structured-data>\n{json.dumps(ld)[:LD_CHARS]}\n</structured-data>\n\n{text}'
    return None if not text.strip() else AttrDict(text=text, url=str(url))

@patch
def research(self:LocalHost, query):
    "The cited corpus fossick assembled: its `digest`, and not the record it assembled it from."
    return str((self._fossick().research(str(query)) or {}).get('digest') or '')

@patch(as_prop=True)
def research_note(self:LocalHost): return 'fossick' if self.web else 'web access is switched off'

LocalHost.THIN_PAGE = 400
LocalHost.READERS = (
        (re.compile(r'https?://(www\.)?github\.com/[^/]+/[^/]+/(blob|raw)/', re.I), 'read_gh_file', {}),
        (re.compile(r'https?://(www\.)?arxiv\.org/(abs|pdf)/', re.I), 'read_arxiv', dict(save_pdf=False, source=True)),
        (re.compile(r'https?://(www\.)?(youtube\.com/watch|youtu\.be/)', re.I), 'read_yt', {}),
    )

### Memory, watches and an API specification

Three groups this host does not implement itself. It holds the backend and forwards to it, and says
so through `without` when there is no backend to forward to. A vault is a
[vishalakshi](https://github.com/vedicreader/vishalakshi) store, opened by whatever built the host.

In [ ]:
#| export
def _needs(host, what):
    "The refusal a forwarding method gives when its backend was never attached."
    return HostError(f'this host has no {what}')

@patch
def memory_search(self:LocalHost, query, limit=8):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.search(str(query), limit=int(limit))

@patch
def memory_tree(self:LocalHost, document=''):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.tree(str(document or ''))

@patch
def memory_read(self:LocalHost, node_id):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.read(str(node_id))

@patch
def memory_topics(self:LocalHost, limit=12):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.topics(limit=int(limit))

@patch
def memory_forget(self:LocalHost, doc_id):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.forget(str(doc_id))

@patch
def remember(self:LocalHost, text, title=None, tags=()):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.remember(str(text), title=title, tags=tuple(tags))

@patch
def ask(self:LocalHost, question, ref=None, instruction='', **kw):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.ask(str(question), ref=ref, instruction=instruction, **kw)

@patch
def watch(self:LocalHost, target, action='remind', every='1d', note=None, **params):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.watch(target, action=action, every=every, note=note, **params)

@patch
def watches(self:LocalHost, due_only=False):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.watches(due_only=bool(due_only))

@patch
def unwatch(self:LocalHost, watch_id): 
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.unwatch(str(watch_id))

@patch
def poll(self:LocalHost):
    if self.vault is None: raise _needs(self, 'vault')
    return self.vault.poll()

In [ ]:
#| export
@patch
def api_load(self:LocalHost, src, name=''):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_load(src, name=name)

@patch
def api_ops(self:LocalHost, group='', name='', match='', limit=None, offset=0):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_ops(group=group, name=name, match=match, limit=limit, offset=offset)

@patch
def api_count(self:LocalHost, group='', name='', match=''):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_count(group=group, name=name, match=match)

@patch
def api_call(self:LocalHost, operation, name='', **params):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_call(operation, name=name, **params)

`@patch` adds a method after the class is built, and `ABCMeta` worked out what was still missing
while it was building. `implemented` asks the question again, by the same rule ABCMeta uses. A
method that nothing ever wrote stays abstract, so a host with a hole in it still refuses to be
built.

In [ ]:
#| export
def implemented(cls):
    "Recompute what `cls` is still missing, after `@patch` filled some of it in."
    names = {m for base in cls.__mro__ for m in getattr(base, '__abstractmethods__', ())}
    cls.__abstractmethods__ = frozenset(
        n for n in names if getattr(getattr(cls, n, None), '__isabstractmethod__', False))
    return cls

implemented(LocalHost)

### What a real host answers

A folder with a little source in it, and a host over it. `index=False` keeps Kosha out of a test
that is about the boundary rather than about retrieval.

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text('def threshold(n):\n    "Half of n."\n    return n // 2\n')
(root/'pkg'/'use.py').write_text('from .sizes import threshold\n\ndef budget(): return threshold(8192)\n')
local = LocalHost([root], index=False)
sorted(local.provides), sorted(local.without)

In [ ]:
test_eq(sorted(local.without), ['api', 'memory', 'watch'])
test_eq(sorted(local.provides), ['code', 'file', 'git', 'notebook', 'session', 'shell', 'web'])
test_eq(local.can('code'), True)
test_eq(local.can('memory'), False)

`provides` is the whole capability answer, and it is honest about a group whose backend was never
attached. Calling into one anyway says which backend is missing rather than raising an
`AttributeError` from inside a forward.

In [ ]:
test_fail(lambda: local.memory_tree(''), contains='no vault')
test_fail(lambda: local.api_ops(), contains='no API specifications')

The sandbox is why this class exists. `check` resolves before it compares, so `..` and a symlink
out of the tree are refused rather than followed.

In [ ]:
test_fail(lambda: local.check('../../etc/passwd'), contains='outside the open folders')
test_fail(lambda: local.check('pkg/nope.py', must_exist=True), contains='no such file')
test_eq(local.check('pkg/sizes.py').name, 'sizes.py')

Reads and writes go through the same chokepoint. `text_at` answers `''` for a file that does not
exist yet, so a diff against a new file is a diff rather than an error.

In [ ]:
local.write('pkg/new.py', 'x = 1\n')
local.read('pkg/new.py'), local.text_at('pkg/nope.py')

In [ ]:
test_eq(local.read('pkg/new.py'), 'x = 1\n')
test_eq(local.text_at('pkg/nope.py'), '')
test_eq((root/'pkg'/'new.py').read_text(), 'x = 1\n')

Symbols come from parsing rather than grepping, so a method is reported at its own depth and under
its qualified name. `peers` finds where the symbol defined at a line is used, which is the useful
call site rather than the definition.

In [ ]:
[(h.symbol, h.line, h.score) for h in local.symbols('pkg/sizes.py')]

In [ ]:
test_eq([h.symbol for h in local.symbols('pkg/sizes.py')], ['threshold'])
peers = local.peers('pkg/sizes.py', 1)
assert any('use.py' in str(h.path) for h in peers), peers

With no index built, `search` is ripgrep, and `search_note` says so. The note is part of the result:
"no matches" and "nothing could be looked up" must not reach the model as the same answer.

In [ ]:
local.search('threshold')[:2], local.search_note

In [ ]:
assert local.search('threshold'), 'ripgrep found nothing'
assert 'fallback' in local.search_note, local.search_note
test_eq(local.search(''), [])

A notebook is two operations rather than a representation. Adding a cell to a path that does not
exist writes the notebook.

In [ ]:
cid = local.nb_add_cell('nb/demo.ipynb', 'x = threshold(4096)\nx')
local.nb_cells('nb/demo.ipynb')

In [ ]:
test_eq(len(local.nb_cells('nb/demo.ipynb')), 1)
test_eq(local.nb_cells('nb/demo.ipynb')[0][0], cid)
assert 'x = threshold(4096)' in local.text_at('nb/demo.ipynb')

The live namespace persists between calls, which is what makes "bind results to new names" a rule
worth briefing an agent on. A trailing expression returns its value, and a failure comes back as
text: a tool that raises ends the turn, and a misspelled name is not the end of a turn.

In [ ]:
local.run_python('import math\nradii = [1, 2, 3]'), local.run_python('areas = [math.pi*r*r for r in radii]\nlen(areas)')

In [ ]:
test_eq(local.run_python('len(areas)'), '3')
assert 'NameError' in local.run_python('no_such_name + 1')
test_eq(local.run_python('print("a side effect")'), 'a side effect')

`inspect_python` runs against a copy of the namespace, so a name it binds is gone when it returns.

The copy is shallow, which is the whole of what this host protects. A binding cannot move. An object
can: `radii.append(99)` reaches the same list the user has. `LocalHost` is the reference host rather
than a sandbox, and `scopes` is `('isolated',)` because a real allowlist policy over a live
namespace belongs to whatever owns the kernel.

In [ ]:
local.inspect_python('doubled = [r*2 for r in radii]\nlen(doubled)'), local.run_python('"doubled" in dir()')

In [ ]:
test_eq(local.inspect_python('doubled = [r*2 for r in radii]\nlen(doubled)'), '3')
test_eq(local.run_python('"doubled" in globals()'), 'False')   # the binding never landed
local.inspect_python('radii.append(99)')
test_eq(local.run_python('len(radii)'), '4')                   # the object did
test_eq(local.scopes, ('isolated',))

A command runs in its own process group under the first open folder, and a failure is an exit code
rather than an exception.

In [ ]:
local.run_cmd('echo hi'), local.run_cmd('exit 3')

In [ ]:
test_eq(local.run_cmd('echo hi'), (0, 'hi\n'))
test_eq(local.run_cmd('exit 3')[0], 3)
test_eq(local.run_cmd('pwd')[1].strip(), str(root))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()